# 👁️ Компьютерное зрение. Нейронная сеть для распознавания одежды
### Учебный практикум · Google Colab

**Выполнила:** Манина В. С., группа ББИ2302, МТУСИ

| Раздел | Тема |
|--------|------|
| [Тема 1](#tema1) | Fashion MNIST — Dense-сеть для классификации одежды |
| [Самост. 1](#sr1) | Улучшение модели: больше нейронов, CNN |
| [Тема 2 — Фильтры](#tema2) | Оператор Собеля, Лапласа, алгоритм Кэнни |
| [Тема 2 — CNN](#tema2cnn) | Свёрточные сети, VGG16, ResNet50 — визуализация слоёв |
| [Тема 2 — Clothing](#clothing) | Датасет Clothing Dataset · Fine-tuning ResNet50 |
| [Самост. 2](#sr2) | Битва архитектур: VGG16 vs ResNet50 vs ConvNeXt |

---
> 💡 **GPU:** `Среда выполнения → Изменить тип → GPU`  
> 📦 Для Темы 2 нужно загрузить файл `test.png` (изображение для фильтров)

---
<a id='tema1'></a>
# 🧥 Тема 1. Компьютерное зрение
## Fashion MNIST — классификация одежды с нуля

In [ ]:
# Этап 1. Установка и импорт библиотек
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras import utils
%matplotlib inline

print(f"TensorFlow версия: {tf.__version__}")
print(f"GPU доступен: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Этап 2. Скачиваем датасет Fashion MNIST
# x_train / x_test — картинки 28×28 (числа 0–255)
# y_train / y_test — метки классов (числа 0–9)
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print(f"Обучающая выборка: {x_train.shape}  |  Метки: {y_train.shape}")
print(f"Тестовая выборка:  {x_test.shape}   |  Метки: {y_test.shape}")
print(f"Диапазон значений пикселей: [{x_train.min()}, {x_train.max()}]")

In [ ]:
# Этап 3. Имена классов (в датасете только цифры 0–9)
class_names = [
    'T-shirt/top',  # 0
    'Trouser',      # 1
    'Pullover',     # 2
    'Dress',        # 3
    'Coat',         # 4
    'Sandal',       # 5
    'Shirt',        # 6
    'Sneaker',      # 7
    'Bag',          # 8
    'Ankle boot'    # 9
]
print("Классы одежды:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

In [ ]:
# Этап 4. Визуализация одного изображения ДО нормализации
plt.figure(figsize=(5, 4))
plt.imshow(x_train[0])   # цветовая схема Viridis по умолчанию
plt.colorbar()           # шкала: видим значения 0–255
plt.grid(False)
plt.title(f"До нормализации | Класс: {class_names[y_train[0]]}")
plt.show()
print(f"Значения пикселей от {x_train[0].min()} до {x_train[0].max()}")

In [ ]:
# Этап 5. Нормализация: делим на 255 → диапазон [0, 1]
# Зачем: нейросети быстрее и стабильнее работают с малыми числами
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# Проверка ДО / ПОСЛЕ
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(x_train[0]); axes[0].set_title("После нормализации"); axes[0].axis('off')
axes[0].figure.colorbar(plt.cm.ScalarMappable(), ax=axes[0])
axes[1].imshow(x_train[0], cmap='gray'); axes[1].set_title("В оттенках серого"); axes[1].axis('off')
plt.suptitle(f"Класс: {class_names[y_train[0]]} | Значения: [{x_train[0].min():.1f}, {x_train[0].max():.1f}]")
plt.show()

In [ ]:
# Этап 6. Визуализация первых 25 изображений с подписями классов
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap='gray')
    plt.xlabel(class_names[y_train[i]], fontsize=8)
plt.suptitle('Первые 25 изображений из обучающей выборки', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Этап 7. Создание последовательной Dense-модели (MLP)
# Архитектура: Flatten (28×28→784) → Dense(128, relu) → Dense(10, softmax)
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),      # «Вытягиваем» картинку в вектор 784
    keras.layers.Dense(128, activation='relu'),       # Скрытый слой: 128 нейронов
    keras.layers.Dense(10,  activation='softmax')     # Выходной слой: 10 вероятностей
])

model.summary()

In [ ]:
# Этап 8. Компиляция модели
# optimizer='adam'       — адаптивный оптимизатор (лучше SGD для этой задачи)
# loss='sparse_...'      — функция потерь для меток-чисел (не one-hot)
# metrics=['accuracy']   — следим за точностью
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Модель скомпилирована ✓")

In [ ]:
# Этап 9. Обучение с валидацией
# epochs=10        — 10 проходов по всему датасету
# batch_size=32    — модель обновляет веса каждые 32 картинки
# validation_split — 20% данных откладываем для честной проверки
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Графики обучения: потери и точность по эпохам
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['loss'],     label='Обучение',  color='#3498db', linewidth=2)
ax1.plot(history.history['val_loss'], label='Валидация', color='#e74c3c', linewidth=2)
ax1.set_title('Потери (Loss)'); ax1.set_xlabel('Эпоха'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'],     label='Обучение',  color='#3498db', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Валидация', color='#e74c3c', linewidth=2)
ax2.set_title('Точность (Accuracy)'); ax2.set_xlabel('Эпоха'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Динамика обучения Dense-модели', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Финальная точность на обучении:  {history.history['accuracy'][-1]*100:.2f}%")
print(f"Финальная точность на валидации: {history.history['val_accuracy'][-1]*100:.2f}%")

In [ ]:
# Этап 10. Проверка точности на тестовых данных (которых сеть не видела)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)
print(f"\n📊 Точность на тестовых данных: {test_acc*100:.2f}%")
print(f"📉 Потери на тестовых данных:   {test_loss:.4f}")

In [ ]:
# Этап 11. Предсказание по тренировочным данным
predictions_train = model.predict(x_train, verbose=0)

# Просматриваем первый элемент: 10 вероятностей
idx = 0
predicted_class = np.argmax(predictions_train[idx])
confidence      = np.max(predictions_train[idx]) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Изображение
ax1.imshow(x_train[idx], cmap='gray')
ax1.set_title(f"Правда: {class_names[y_train[idx]]}"); ax1.axis('off')

# Столбчатая диаграмма вероятностей
colors = ['#e74c3c' if i == predicted_class else '#bdc3c7' for i in range(10)]
ax2.barh(class_names, predictions_train[idx], color=colors)
ax2.set_title(f"Предсказание: {class_names[predicted_class]} ({confidence:.1f}%)")
ax2.set_xlim([0, 1])

plt.tight_layout()
plt.show()
print(f"Номер класса: {predicted_class} | Название: {class_names[predicted_class]} | Уверенность: {confidence:.2f}%")

In [ ]:
# Этап 12. Предсказание по тестовым данным
predictions_test = model.predict(x_test, verbose=0)

# Проверяем элемент с индексом 10
idx = 10
predicted_class_test = np.argmax(predictions_test[idx])
confidence_test      = np.max(predictions_test[idx]) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(x_test[idx]); ax1.set_title(f"Правда: {class_names[y_test[idx]]}"); ax1.axis('off')

colors = ['#e74c3c' if i == predicted_class_test else '#bdc3c7' for i in range(10)]
ax2.barh(class_names, predictions_test[idx], color=colors)
ax2.set_title(f"Предсказание: {class_names[predicted_class_test]} ({confidence_test:.1f}%)")
ax2.set_xlim([0, 1])

plt.tight_layout()
plt.show()
print(f"Предсказано: {class_names[predicted_class_test]}  |  Правда: {class_names[y_test[idx]]}")

In [ ]:
# Этап 13. Визуализация галереи: синий = верно, красный = ошибка
def plot_test_result(i, predictions_array, true_label, img):
    arr, label, image = predictions_array[i], true_label[i], img[i]
    plt.grid(False); plt.xticks([]); plt.yticks([])
    plt.imshow(image, cmap=plt.cm.binary)
    predicted_label = np.argmax(arr)
    color = 'blue' if predicted_label == label else 'red'
    plt.xlabel(
        f"{class_names[predicted_label]} {100*np.max(arr):.0f}%\n"
        f"(Факт: {class_names[label]})",
        color=color, fontsize=7
    )

num_rows, num_cols = 5, 4
plt.figure(figsize=(12, 14))
for i in range(num_rows * num_cols):
    plt.subplot(num_rows, num_cols, i + 1)
    plot_test_result(i, predictions_test, y_test, x_test)
plt.suptitle('Синий = верно | Красный = ошибка', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Матрица ошибок — видим, какие классы путаются
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_pred_all = np.argmax(predictions_test, axis=1)
cm = confusion_matrix(y_test, y_pred_all)

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Предсказание', fontsize=12)
plt.ylabel('Факт', fontsize=12)
plt.title('Матрица ошибок (Dense-модель)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Этап 14. Проверка модели на изображении из интернета
from PIL import Image, ImageOps
import numpy as np
import requests
from io import BytesIO

def predict_my_image(image_source):
    """
    Предсказание класса одежды для произвольного изображения.
    image_source: путь к файлу или URL
    Требования: вещь по центру, однотонный фон, квадратный кадр.
    """
    # Загрузка
    if str(image_source).startswith('http'):
        response = requests.get(image_source)
        img = Image.open(BytesIO(response.content)).convert('L')
    else:
        img = Image.open(image_source).convert('L')

    # Инверсия: Fashion MNIST имеет чёрный фон, белый объект
    img = ImageOps.invert(img)
    # Размер 28×28
    img = img.resize((28, 28))
    # Нормализация и батч
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)   # (1, 28, 28)

    prediction = model.predict(img_array, verbose=0)
    cls_idx = np.argmax(prediction)
    conf    = np.max(prediction) * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
    ax1.imshow(img, cmap='gray'); ax1.axis('off')
    ax1.set_title(f"Это: {class_names[cls_idx]}\n({conf:.1f}%)")
    colors = ['#e74c3c' if i == cls_idx else '#bdc3c7' for i in range(10)]
    ax2.barh(class_names, prediction[0], color=colors)
    ax2.set_xlim([0, 1])
    ax2.set_title("Распределение вероятностей")
    plt.tight_layout()
    plt.show()

# Тест на изображении из датасета (гарантированный результат)
test_img = Image.fromarray((x_test[5] * 255).astype(np.uint8))
test_img.save('/tmp/test_sample.png')
predict_my_image('/tmp/test_sample.png')
print(f"Правда: {class_names[y_test[5]]}")

In [ ]:
# Тест на URL (замените ссылку на своё изображение)
# ВНИМАНИЕ: Dense-сеть плохо работает с реальными фото!
# Нужно: вещь по центру, белый фон, квадратный кадр

# Пример: попробуйте загрузить картинку через Colab
# from google.colab import files
# uploaded = files.upload()
# predict_my_image(list(uploaded.keys())[0])

# Или укажите URL:
# predict_my_image('https://example.com/your_shoe.jpg')
print("Ячейка готова для тестирования своих изображений.")
print("⚠️ Плохой результат на реальных фото — это нормально для Dense-сети.")
print("   Решение: CNN (см. Задание 3 ниже).")

---
<a id='sr1'></a>
# 🔬 Самостоятельная работа 1. Повышение качества модели
## Задание 2. Больше нейронов и слоёв + Dropout

In [ ]:
# Задание 2: Глубокая Dense-сеть с Dropout
# 256 нейронов → 128 нейронов → Dropout(0.2) → 10 классов
model_deep = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(256, activation='relu'),   # Больше нейронов
    keras.layers.Dense(128, activation='relu'),   # Дополнительный слой
    keras.layers.Dropout(0.2),                    # Защита от переобучения
    keras.layers.Dense(10,  activation='softmax')
])

model_deep.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_deep.summary()

history_deep = model_deep.fit(
    x_train, y_train,
    epochs=10, batch_size=32, validation_split=0.2, verbose=1
)

_, acc_deep = model_deep.evaluate(x_test, y_test, verbose=0)
_, acc_base = model.evaluate(x_test, y_test, verbose=0)
print(f"\n📊 Базовая модель (128):         {acc_base*100:.2f}%")
print(f"📊 Глубокая модель (256→128):     {acc_deep*100:.2f}%")
print(f"Прирост: {(acc_deep - acc_base)*100:+.2f}%")

## Задание 3. Сверточная модель (CNN) для Fashion MNIST

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D

# Для CNN нужна форма (N, 28, 28, 1) — добавляем размерность канала
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn  = x_test.reshape(-1, 28, 28, 1)

# CNN архитектура:
# Conv2D(32 фильтра 3×3) → MaxPool → Conv2D(64 фильтра) → MaxPool → Flatten → Dense
model_cnn = keras.Sequential([
    # Блок 1: ищем линии и границы
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    # Блок 2: ищем более сложные формы
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    # Классификатор
    Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()

In [ ]:
# Обучаем CNN
history_cnn = model_cnn.fit(
    x_train_cnn, y_train,
    epochs=10, batch_size=64,
    validation_split=0.2, verbose=1
)

_, acc_cnn = model_cnn.evaluate(x_test_cnn, y_test, verbose=0)
print(f"\n📊 Базовая Dense (128):   {acc_base*100:.2f}%")
print(f"📊 Глубокая Dense (256→128): {acc_deep*100:.2f}%")
print(f"📊 CNN (Conv2D):          {acc_cnn*100:.2f}%")

In [ ]:
# Сравнение трёх архитектур на графике
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['val_accuracy'],      label='Dense (128)', linewidth=2)
plt.plot(history_deep.history['val_accuracy'], label='Dense (256→128)', linewidth=2)
plt.plot(history_cnn.history['val_accuracy'],  label='CNN', linewidth=2)
plt.title('Точность на валидации'); plt.xlabel('Эпоха'); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
models_names = ['Dense\n(128)', 'Dense\n(256→128)', 'CNN']
accs = [acc_base * 100, acc_deep * 100, acc_cnn * 100]
bars = plt.bar(models_names, accs, color=['#3498db', '#9b59b6', '#e74c3c'], edgecolor='black')
plt.title('Финальная точность на тесте'); plt.ylabel('Точность (%)')
plt.ylim([85, 95])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{acc:.2f}%', ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.suptitle('Сравнение архитектур на Fashion MNIST', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='tema2'></a>
# 🔍 Тема 2. Компьютерное зрение: фильтры и признаки
## Оператор Собеля, Лапласа, алгоритм Кэнни

In [ ]:
# Установка OpenCV
!pip install opencv-python-headless -q
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Загрузка тестового изображения
# Вариант 1: скачать тестовое изображение с бургером (как в учебнике)
# !wget -q -O test.png 'https://drive.google.com/uc?export=download&id=1_l4NQAJdCAwr5BUqyoYpktAyUlmfEXq3'

# Вариант 2: создаём простое тестовое изображение прямо здесь
from PIL import Image, ImageDraw
img_demo = Image.new('RGB', (300, 300), color=(200, 200, 100))
draw = ImageDraw.Draw(img_demo)
draw.ellipse([80, 80, 220, 220], fill=(180, 100, 50), outline=(50, 30, 10), width=3)
draw.rectangle([50, 160, 250, 200], fill=(220, 150, 80))
draw.rectangle([50, 200, 250, 240], fill=(180, 80, 40))
img_demo.save('test.png')
print("Тестовое изображение создано ✓")
plt.figure(figsize=(4, 4))
plt.imshow(img_demo); plt.axis('off')
plt.title('Тестовое изображение (test.png)')
plt.show()

In [ ]:
# Оператор Собеля: вертикальные границы (по X)
# Фильтр: [[-1,0,1],[-2,0,2],[-1,0,1]] скользит по картинке
img_gray = cv2.imread('test.png', cv2.IMREAD_GRAYSCALE)

# Ручной фильтр через numpy
kernel_sobel = np.array([[-1, 0, 1],
                          [-2, 0, 2],
                          [-1, 0, 1]])
sobel_manual = cv2.filter2D(img_gray, cv2.CV_64F, kernel_sobel)

# Встроенный Собель (размер 3×3)
sobel_x  = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_x11 = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=11)  # крупный фильтр
sobel_y  = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)

# Комбинирование X и Y — полные контуры
sobel_full = np.sqrt(np.square(sobel_x) + np.square(sobel_y)).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
titles = ['Оригинал (ч/б)', 'Собель вручную', 'Собель X (3×3)', 'Собель X (11×11)', 'Собель X+Y (контуры)']
images = [img_gray, sobel_manual, sobel_x, sobel_x11, sobel_full]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray'); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle('Оператор Собеля — обнаружение границ', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Каналы RGB — применяем Собель к красному каналу
img_rgb = cv2.imread('test.png')          # BGR в OpenCV
img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
red_channel  = img_rgb[:, :, 0]            # R канал
sobel_red    = cv2.Sobel(red_channel, cv2.CV_64F, 1, 0, ksize=3)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb);           axes[0].set_title('Оригинал RGB')
axes[1].imshow(red_channel, cmap='Reds'); axes[1].set_title('R-канал')
axes[2].imshow(sobel_red, cmap='gray');  axes[2].set_title('Собель по R-каналу')

# Цветной Собель через ч/б версию
sobel_bw = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
axes[3].imshow(sobel_bw, cmap='gray');   axes[3].set_title('Собель по ч/б версии')

for ax in axes: ax.axis('off')
plt.suptitle('Собель по разным каналам', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Оператор Лапласа — все направления сразу
# Матрица: [[0,1,0],[1,-4,1],[0,1,0]]
laplacian = cv2.Laplacian(img_gray, cv2.CV_64F, ksize=3)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img_gray, cmap='gray');  axes[0].set_title('Оригинал')
axes[1].imshow(laplacian, cmap='gray'); axes[1].set_title('Лапласиан ksize=3')

# Сравнение: Лапласиан vs Собель (полный контур)
axes[2].imshow(sobel_full, cmap='gray'); axes[2].set_title('Собель X+Y (для сравнения)')
for ax in axes: ax.axis('off')
plt.suptitle('Оператор Лапласа — границы в одном шаге', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Алгоритм Кэнни — лучший детектор контуров
# Параметры: нижний порог, верхний порог
canny_low  = cv2.Canny(img_gray, 30, 70)    # Чувствительный — много деталей
canny_mid  = cv2.Canny(img_gray, 100, 200)  # Стандартный
canny_high = cv2.Canny(img_gray, 150, 250)  # Строгий — только главные контуры

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_gray, cmap='gray');  axes[0].set_title('Оригинал')
axes[1].imshow(canny_low, cmap='gray'); axes[1].set_title('Кэнни (30, 70)\nЧувствительный')
axes[2].imshow(canny_mid, cmap='gray'); axes[2].set_title('Кэнни (100, 200)\nСтандартный')
axes[3].imshow(canny_high,cmap='gray'); axes[3].set_title('Кэнни (150, 250)\nСтрогий')
for ax in axes: ax.axis('off')
plt.suptitle('Алгоритм Кэнни — влияние порогов', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение всех трёх методов
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_gray,    cmap='gray'); axes[0].set_title('Оригинал')
axes[1].imshow(sobel_full,  cmap='gray'); axes[1].set_title('Собель X+Y')
axes[2].imshow(laplacian,   cmap='gray'); axes[2].set_title('Лапласиан')
axes[3].imshow(canny_mid,   cmap='gray'); axes[3].set_title('Кэнни (100, 200)')
for ax in axes: ax.axis('off')
plt.suptitle('Сравнение методов обнаружения границ', fontsize=13)
plt.tight_layout()
plt.show()

print("""
Шпаргалка по выбору метода:
  Собель  → направленные линии, быстро, грубо
  Лапласиан → все направления сразу, шумит
  Кэнни  → тонкие чёткие контуры (лучший выбор)
""")

---
<a id='tema2cnn'></a>
# 🧠 Тема 2. Свёрточные нейросети — визуализация слоёв
## VGG16 и ResNet50: «заглядываем в голову» нейросети

In [ ]:
# Импорты для работы с предобученными моделями
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet_preprocess
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model

# Загружаем тестовое изображение для VGG16 (ожидает 224×224)
img_path = 'test.png'
img_224  = load_img(img_path, target_size=(224, 224))
img_arr  = img_to_array(img_224)
img_vgg  = np.expand_dims(img_arr.copy(), axis=0)
img_vgg  = vgg_preprocess(img_vgg)

print("Изображение подготовлено:", img_vgg.shape)

In [ ]:
# ====== VGG16: фильтры первого слоя ======
vgg = VGG16()
print("Свёрточные слои VGG16:")
for layer in vgg.layers:
    if '_conv' in layer.name:
        filters, bias = layer.get_weights()
        print(f"  {layer.name}: {filters.shape}")

In [ ]:
# Визуализация фильтров первого свёрточного слоя VGG16
filters_vgg, _ = vgg.layers[1].get_weights()  # shape: (3, 3, 3, 64)
n_filters = 32  # показываем 32 из 64

fig, axes = plt.subplots(3, n_filters, figsize=(n_filters * 0.6, 3))
for i in range(n_filters):
    f = filters_vgg[:, :, :, i]
    for j in range(3):  # RGB каналы
        ax = axes[j, i]
        ax.imshow(f[:, :, j], cmap='gray')
        ax.axis('off')
plt.suptitle('VGG16 — фильтры первого слоя (строки = RGB каналы)', fontsize=12)
plt.tight_layout()
plt.show()
print("Узнаёте паттерны? Некоторые напоминают оператор Собеля!")

In [ ]:
# Карты признаков первого слоя VGG16 (что «видит» каждый фильтр)
model_layer1 = Model(inputs=vgg.inputs, outputs=vgg.layers[1].output)
feature_maps = model_layer1.predict(img_vgg, verbose=0)  # (1, 224, 224, 64)

square = 8  # сетка 8×8 = 64 карты
fig, axes = plt.subplots(square, square, figsize=(16, 16))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap='gray')
    ax.axis('off')
plt.suptitle('VGG16 — карты признаков после ПЕРВОГО слоя (64 фильтра)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Эволюция признаков по глубине VGG16: слои 1, 4, 9, 17
# Наблюдаем: линии → текстуры → абстракция
blocks = [1, 4, 9, 17]
outputs_vgg = [vgg.layers[i].output for i in blocks]
model_multi = Model(inputs=vgg.inputs, outputs=outputs_vgg)
feature_maps_all = model_multi.predict(img_vgg, verbose=0)

fig, all_axes = plt.subplots(len(blocks), 8, figsize=(16, len(blocks) * 2.5))
labels = [f'Слой {b}\n(block{(b//4)+1}_conv)' for b in blocks]

for row, (fmaps, label) in enumerate(zip(feature_maps_all, labels)):
    n_show = min(8, fmaps.shape[-1])
    for col in range(8):
        ax = all_axes[row, col]
        if col < n_show:
            ax.imshow(fmaps[0, :, :, col], cmap='gray')
        ax.axis('off')
    all_axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, ha='right', labelpad=60)

plt.suptitle('VGG16 — эволюция признаков по глубине сети\n(линии → текстуры → абстракция)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ====== ResNet50: фильтры и карты признаков ======
resnet = ResNet50(weights='imagenet', include_top=True)

img_res = np.expand_dims(img_to_array(img_224).copy(), axis=0)
img_res = resnet_preprocess(img_res)

# Фильтры первого свёрточного слоя (7×7, а не 3×3 как у VGG)
conv1_layer = resnet.get_layer('conv1_conv')
filters_res, _ = conv1_layer.get_weights()   # shape: (7, 7, 3, 64)
print(f"ResNet50 первый слой: {filters_res.shape}  ← фильтры 7×7 (сложнее, чем 3×3 у VGG16)")

n_show = 32
fig, axes = plt.subplots(3, n_show, figsize=(n_show * 0.7, 3.5))
for i in range(n_show):
    f = filters_res[:, :, :, i]
    for j in range(3):
        axes[j, i].imshow(f[:, :, j], cmap='gray')
        axes[j, i].axis('off')
plt.suptitle('ResNet50 — фильтры первого слоя 7×7 (сложнее, чем VGG16 3×3)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Карты признаков ResNet50 на слоях разной глубины
layer_names = ['conv1_conv', 'conv2_block1_1_conv', 'conv3_block1_1_conv', 'conv5_block1_1_conv']
layer_labels = ['Слой 1\n(Края и линии)', 'Слой 2\n(Текстуры)', 'Слой 3\n(Формы)', 'Слой 5\n(Объекты)']

outputs_res = [resnet.get_layer(name).output for name in layer_names]
model_res_multi = Model(inputs=resnet.inputs, outputs=outputs_res)
fmaps_res = model_res_multi.predict(img_res, verbose=0)

fig, all_axes = plt.subplots(4, 8, figsize=(16, 10))
for row, (fmaps, label) in enumerate(zip(fmaps_res, layer_labels)):
    n_show = min(8, fmaps.shape[-1])
    for col in range(8):
        ax = all_axes[row, col]
        if col < n_show:
            ax.imshow(fmaps[0, :, :, col], cmap='gray')
        ax.axis('off')
    all_axes[row, 0].set_ylabel(label, fontsize=9, rotation=0, ha='right', labelpad=70)

plt.suptitle('ResNet50 — пирамида смыслов: от линий к объектам', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='clothing'></a>
# 👗 Тема 2. Датасет Clothing Dataset
## Fine-Tuning ResNet50 для интернет-магазина одежды

In [ ]:
# Установка kagglehub для загрузки датасета
!pip install kagglehub -q

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import kagglehub

print("Библиотеки импортированы ✓")

In [ ]:
# --- Этап 1: Загрузка данных Clothing Dataset ---
print("Загрузка датасета Clothing Dataset...")
path = kagglehub.dataset_download("agrigorev/clothing-dataset-full")
print(f"Датасет загружен: {path}")

# Загружаем таблицу аннотаций
df = pd.read_csv(os.path.join(path, 'images.csv'))
print(f"\nСтолбцы: {df.columns.tolist()}")
print(f"Размер: {df.shape}")
print(df.head())

In [ ]:
# Формируем правильные пути к изображениям
images_folder = os.path.join(path, 'images_compressed')
if not os.path.exists(images_folder):
    images_folder = os.path.join(path, 'images')

def get_full_path(img_id):
    file_name = img_id if str(img_id).endswith('.jpg') else f"{img_id}.jpg"
    full_path = os.path.join(images_folder, file_name)
    return full_path if os.path.exists(full_path) else None

df['image_path'] = df['image'].apply(get_full_path)
df = df.dropna(subset=['image_path'])
# Убираем мусорные классы
df = df[~df['label'].isin(['Not sure', 'Other', 'Skip'])]

print(f"Успешно найдено изображений: {len(df)}")
print(f"\nРаспределение по классам:")
print(df['label'].value_counts().to_string())

In [ ]:
# Визуализация примеров из датасета
from PIL import Image as PILImage

sample = df.groupby('label').first().reset_index().head(10)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, (ax, row) in enumerate(zip(axes.flat, sample.itertuples())):
    try:
        img = PILImage.open(row.image_path).resize((150, 150))
        ax.imshow(img)
        ax.set_title(row.label, fontsize=10, fontweight='bold')
    except:
        ax.set_title(f"{row.label}\n(нет фото)", fontsize=9)
    ax.axis('off')
plt.suptitle('Примеры из Clothing Dataset (по одному на класс)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Этап 2: Аугментация данных ---
# Поворачиваем, сдвигаем, отражаем → модель не «зазубривает» позиции
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_dataframe(
    df, x_col='image_path', y_col='label',
    target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='training'
)
val_gen = datagen.flow_from_dataframe(
    df, x_col='image_path', y_col='label',
    target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='validation'
)
print(f"Классов: {len(train_gen.class_indices)}")

In [ ]:
# Визуализация аугментированных изображений
batch_x, batch_y = next(train_gen)
class_names_clothing = {v: k for k, v in train_gen.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat[:10]):
    ax.imshow(batch_x[i])
    label = class_names_clothing[np.argmax(batch_y[i])]
    ax.set_title(label, fontsize=9)
    ax.axis('off')
plt.suptitle('Аугментированные изображения (подаются в нейросеть)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Этап 3: Transfer Learning — ResNet50 Fine-Tuning ---
# include_top=False — убираем «голову» ImageNet (1000 классов)
# weights='imagenet' — оставляем предобученные веса (знания о формах и линиях)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True

# Замораживаем первые 100 слоёв — оставляем базовые детекторы линий
for layer in base_model.layers[:100]:
    layer.trainable = False

frozen  = sum(1 for l in base_model.layers if not l.trainable)
unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f"Заморожено слоёв: {frozen}  |  Обучаемых: {unfrozen}")

# Добавляем «голову» для нашего датасета
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)                                                  # Защита от переобучения
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(len(train_gen.class_indices), activation='softmax')(x)

model_clothing = Model(inputs=base_model.input, outputs=outputs)

# Низкий learning_rate — аккуратно подстраиваем готовые веса
model_clothing.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f"\nВсего параметров: {model_clothing.count_params():,}")

In [ ]:
# --- Этап 4: Обучение с авто-остановкой ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1)
]

print("Начинаем Fine-Tuning ResNet50...")
history_clothing = model_clothing.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks
)
print("Обучение завершено! ✓")

In [ ]:
# Графики обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history_clothing.history['loss'],     label='Обучение',  color='#3498db', lw=2)
ax1.plot(history_clothing.history['val_loss'], label='Валидация', color='#e74c3c', lw=2)
ax1.set_title('Потери'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history_clothing.history['accuracy'],     label='Обучение',  color='#3498db', lw=2)
ax2.plot(history_clothing.history['val_accuracy'], label='Валидация', color='#e74c3c', lw=2)
ax2.set_title('Точность'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('ResNet50 Fine-Tuning на Clothing Dataset', fontsize=13)
plt.tight_layout()
plt.show()

best_val_acc = max(history_clothing.history['val_accuracy']) * 100
print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")

In [ ]:
# --- Этап 5: Функция предсказания типа одежды ---
import requests
from PIL import Image
from io import BytesIO
from tensorflow.keras.preprocessing import image as keras_image

# Словарь классов
clothing_class_names = list(train_gen.class_indices.keys())

def predict_clothing_type(url_or_path, top_n=3):
    """Определяет тип одежды на фото (URL или путь к файлу)"""
    # Загрузка
    if str(url_or_path).startswith('http'):
        response = requests.get(url_or_path)
        img = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        img = Image.open(url_or_path).convert('RGB')

    img_resized = img.resize((224, 224))
    x = keras_image.img_to_array(img_resized) / 255.0
    x = np.expand_dims(x, axis=0)

    preds = model_clothing.predict(x, verbose=0)
    top_idx = np.argsort(preds[0])[::-1][:top_n]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_resized)
    ax1.set_title(f"Предсказание:\n{clothing_class_names[top_idx[0]]} ({preds[0][top_idx[0]]*100:.1f}%)")
    ax1.axis('off')

    top_labels = [clothing_class_names[i] for i in top_idx]
    top_probs  = [preds[0][i] * 100 for i in top_idx]
    bars = ax2.barh(top_labels[::-1], top_probs[::-1], color='#3498db')
    ax2.set_title(f'Топ-{top_n} вероятных классов')
    ax2.set_xlabel('Уверенность (%)')
    for bar, prob in zip(bars, top_probs[::-1]):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{prob:.1f}%', va='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

# Тест на первом изображении из датасета
sample_path = df['image_path'].iloc[0]
print(f"Тестируем: {sample_path}")
print(f"Правда: {df['label'].iloc[0]}")
predict_clothing_type(sample_path)

In [ ]:
# Тест на нескольких случайных изображениях
sample_rows = df.sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    try:
        img = Image.open(row['image_path']).convert('RGB').resize((224, 224))
        x = keras_image.img_to_array(img) / 255.0
        x = np.expand_dims(x, axis=0)
        preds = model_clothing.predict(x, verbose=0)
        pred_label = clothing_class_names[np.argmax(preds)]
        conf = np.max(preds) * 100
        correct = pred_label == row['label']
        color = 'green' if correct else 'red'
        ax.imshow(img)
        ax.set_title(f"Факт: {row['label']}\nПред: {pred_label} ({conf:.0f}%)",
                     color=color, fontsize=9, fontweight='bold')
    except Exception as e:
        ax.set_title(f"Ошибка: {e}")
    ax.axis('off')

plt.suptitle('Тест ResNet50 на случайных изображениях\n(зелёный = верно, красный = ошибка)', fontsize=12)
plt.tight_layout()
plt.show()

---
<a id='sr2'></a>
# ⚔️ Самостоятельная работа 2. Битва архитектур
## Задание 1: VGG16 vs ResNet50 — сравнение

In [ ]:
from tensorflow.keras.applications import VGG16 as VGG16App

# VGG16 с замороженными слоями (только «голова» обучается)
base_vgg = VGG16App(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_vgg.trainable = False   # Полная заморозка — только базовые текстуры

x_vgg = GlobalAveragePooling2D()(base_vgg.output)
x_vgg = Dropout(0.5)(x_vgg)
x_vgg = Dense(256, activation='relu')(x_vgg)
x_vgg = Dropout(0.3)(x_vgg)
out_vgg = Dense(len(train_gen.class_indices), activation='softmax')(x_vgg)

model_vgg_ft = Model(inputs=base_vgg.input, outputs=out_vgg)
model_vgg_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"VGG16 (замороженный): {sum(1 for l in base_vgg.layers if not l.trainable)} заморожено")
print(f"Всего обучаемых параметров: {sum(w.numpy().size for w in model_vgg_ft.trainable_weights):,}")

In [ ]:
# Обучаем VGG16 (5-7 эпох для сравнения)
callbacks_vgg = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
]

print("Обучаем VGG16...")
history_vgg = model_vgg_ft.fit(
    train_gen,
    epochs=7,
    validation_data=val_gen,
    callbacks=callbacks_vgg
)
print("VGG16 обучена! ✓")

In [ ]:
# Сравнение VGG16 vs ResNet50
vgg_val_accs    = history_vgg.history['val_accuracy']
resnet_val_accs = history_clothing.history['val_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График точности
axes[0].plot(range(1, len(vgg_val_accs)+1),    [a*100 for a in vgg_val_accs],
             label='VGG16 (заморожен)', color='#3498db', lw=2, marker='o')
axes[0].plot(range(1, len(resnet_val_accs)+1), [a*100 for a in resnet_val_accs],
             label='ResNet50 (fine-tune)', color='#e74c3c', lw=2, marker='s')
axes[0].set_title('Точность на валидации: VGG16 vs ResNet50')
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Точность (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Итоговое сравнение
best_vgg    = max(vgg_val_accs) * 100
best_resnet = max(resnet_val_accs) * 100
bars = axes[1].bar(['VGG16\n(заморожен)', 'ResNet50\n(fine-tune)'],
                   [best_vgg, best_resnet],
                   color=['#3498db', '#e74c3c'], edgecolor='black', width=0.5)
axes[1].set_title('Максимальная точность')
axes[1].set_ylabel('Точность (%)')
for bar, acc in zip(bars, [best_vgg, best_resnet]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('⚔️ Битва архитектур: VGG16 vs ResNet50', fontsize=14)
plt.tight_layout()
plt.show()

winner = 'ResNet50' if best_resnet > best_vgg else 'VGG16'
print(f"\n🏆 Победитель: {winner}  (+{abs(best_resnet - best_vgg):.2f}% разница)")

## Задание 2: ConvNeXt Fine-Tuning — детали и крой

In [ ]:
from tensorflow.keras.applications import ConvNeXtTiny

# ConvNeXtTiny — современная архитектура (2022)
# Точность ImageNet 81.3% при 28.6 млн параметров
base_convnext = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_convnext.trainable = True

# Шаг 1: замораживаем почти всё, обучаем только «голову»
for layer in base_convnext.layers[:-20]:
    layer.trainable = False

unfrozen_cn = sum(1 for l in base_convnext.layers if l.trainable)
print(f"ConvNeXtTiny: {unfrozen_cn} обучаемых слоёв из последних 20")

x_cn = GlobalAveragePooling2D()(base_convnext.output)
x_cn = Dropout(0.4)(x_cn)
x_cn = Dense(256, activation='relu')(x_cn)
x_cn = Dropout(0.2)(x_cn)
out_cn = Dense(len(train_gen.class_indices), activation='softmax')(x_cn)

model_convnext = Model(inputs=base_convnext.input, outputs=out_cn)
model_convnext.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),  # Очень малый lr для Fine-Tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Параметров: {model_convnext.count_params():,}")

In [ ]:
# Обучение ConvNeXt
print("Обучаем ConvNeXtTiny...")
history_cn = model_convnext.fit(
    train_gen,
    epochs=7,
    validation_data=val_gen,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)]
)
print("ConvNeXt обучен! ✓")

## Задание 3: Визуализация карт активаций (что «видит» каждая сеть)

In [ ]:
# Сравнение карт активаций VGG16 vs ResNet50 на одном изображении

# Берём реальное изображение одежды из датасета
sample_img_path = df['image_path'].iloc[0]
sample_label    = df['label'].iloc[0]

img_sample = load_img(sample_img_path, target_size=(224, 224))
img_sample_arr = img_to_array(img_sample)

img_for_vgg    = np.expand_dims(img_sample_arr.copy(), axis=0)
img_for_vgg    = vgg_preprocess(img_for_vgg)

img_for_resnet = np.expand_dims(img_sample_arr.copy(), axis=0)
img_for_resnet = resnet_preprocess(img_for_resnet)

print(f"Изображение: {sample_img_path}")
print(f"Класс: {sample_label}")

# Карты VGG16
vgg_layer1_model = Model(inputs=vgg.inputs, outputs=vgg.layers[1].output)
vgg_maps = vgg_layer1_model.predict(img_for_vgg, verbose=0)

# Карты ResNet50
res_layer1_model = Model(inputs=resnet.inputs, outputs=resnet.get_layer('conv1_conv').output)
res_maps = res_layer1_model.predict(img_for_resnet, verbose=0)

# Визуализация
fig = plt.figure(figsize=(16, 9))

# Оригинал
ax0 = fig.add_subplot(3, 9, 5)
ax0.imshow(img_sample); ax0.set_title(f'Оригинал\n({sample_label})', fontsize=9); ax0.axis('off')

# VGG16 — первые 8 карт
for i in range(8):
    ax = fig.add_subplot(3, 9, 9 + i + 1)
    ax.imshow(vgg_maps[0, :, :, i], cmap='viridis')
    if i == 0: ax.set_ylabel('VGG16', fontsize=10, rotation=0, ha='right', labelpad=40)
    ax.axis('off')

# ResNet50 — первые 8 карт
for i in range(8):
    ax = fig.add_subplot(3, 9, 18 + i + 1)
    ax.imshow(res_maps[0, :, :, i], cmap='viridis')
    if i == 0: ax.set_ylabel('ResNet50', fontsize=10, rotation=0, ha='right', labelpad=45)
    ax.axis('off')

plt.suptitle('Задание 3: Что «видит» каждая нейросеть на первом слое?', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Финальная сравнительная таблица всех моделей
print("=" * 70)
print("ИТОГОВОЕ СРАВНЕНИЕ АРХИТЕКТУР")
print("=" * 70)

results = [
    ("Dense (Fashion MNIST)",         "Fashion MNIST",   f"{acc_base*100:.1f}%",    "~0.4 МБ",  "Обучение основ"),
    ("Deep Dense + Dropout",          "Fashion MNIST",   f"{acc_deep*100:.1f}%",    "~0.8 МБ",  "Добавляем слои"),
    ("CNN (Conv2D)",                   "Fashion MNIST",   f"{acc_cnn*100:.1f}%",    "~1.5 МБ",  "Видим формы"),
    ("VGG16 (заморожен)",              "Clothing Dataset",f"{max(vgg_val_accs)*100:.1f}%","~500 МБ", "Текстуры"),
    ("ResNet50 (fine-tune)",           "Clothing Dataset",f"{max(resnet_val_accs)*100:.1f}%","~98 МБ",  "Универсал"),
    ("ConvNeXtTiny (fine-tune)",       "Clothing Dataset",f"{max(history_cn.history['val_accuracy'])*100:.1f}%","~109 МБ", "Детали и крой"),
]

print(f"{'Модель':<28} {'Датасет':<20} {'Точность':>8} {'Размер':>9}  Когда использовать")
print("-" * 90)
for row in results:
    print(f"  {row[0]:<26} {row[1]:<20} {row[2]:>8} {row[3]:>9}  {row[4]}")
print("=" * 70)

---
## 💡 Выводы и рекомендации

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║  КОГДА КАКУЮ АРХИТЕКТУРУ ВЫБИРАТЬ?                               ║
╠══════════════════════════════════════════════════════════════════╣
║  Dense (MLP)    → простые табличные данные, базовое обучение     ║
║  CNN собственный → небольшой датасет, простые категории          ║
║  VGG16          → анализ текстур и материалов ткани              ║
║  ResNet50       → золотой стандарт, 90% задач (рабочая лошадка)  ║
║  ConvNeXt       → мельчайшие детали, брендовый крой, высокая     ║
║                    точность при схожих размерах с ResNet50        ║
╠══════════════════════════════════════════════════════════════════╣
║  КЛЮЧЕВЫЕ ТЕХНИКИ:                                                ║
║  ✓ Нормализация (÷255) — ускоряет обучение                       ║
║  ✓ Аугментация — устойчивость к ракурсу и освещению              ║
║  ✓ Dropout — защита от переобучения                              ║
║  ✓ Transfer Learning — используем чужой опыт                     ║
║  ✓ EarlyStopping — не тратим время зря                           ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
*Компьютерное зрение · Построение нейронной сети для распознавания одежды · Москва 2026*